# 部署已训练的 pi_0 策略

<img src="./media/rollout2.gif" width="480" height="360">

在仿真中部署已训练策略。


In [ ]:
!pip install pytest
!pip install transformers==4.50.3


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


# 训练并部署 pi_0


### 【可选】下载数据集


In [2]:
'''
If you want to use the collected dataset, please download it from Hugging Face.
'''
!git clone https://huggingface.co/datasets/Jeongeun/omy_pnp_language

fatal: destination path 'datawhale_eai_pnp_language' already exists and is not an empty directory.


## 第 1 步：修改配置文件 `pi0_datawhale_eai.yaml`


`pi0_datawhale_eai.yaml` 文件
```
dataset:
  repo_id: datawhale_eai_pnp
  root: ./datawhale_eai_pnp
policy:
  type : pi0
  chunk_size: 5
  n_action_steps: 5
save_checkpoint: true
output_dir: ./ckpt/pi0_datawhale_eai
batch_size: 16
job_name : pi0_datawhale_eai
resume: false
seed : 42
num_workers: 8
steps: 20_000
eval_freq: -1 # No evaluation
log_freq: 50
save_checkpoint: true
save_freq: 5_000
use_policy_training_preset: true
  
wandb:
  enable: true
  project: pi0_datawhale_eai
  entity: <YOUR ENTITY for wandb>
  disable_artifact: true
```


## 第 2 步：训练模型
代码在 A100 上测试通过。


In [ ]:
!python train_model.py --config_path pi0_datawhale_eai.yaml

INFO 2025-06-28 20:05:05 ils/utils.py:46 Cuda backend detected, using cuda.
WARNING 2025-06-28 20:05:05 /policies.py:67 Device 'None' is not available. Switching to 'cuda'.
Traceback (most recent call last):
  File "/home/jeongeun/Dropbox/code/vla_prj/lerobot-mujoco-tutorial/train_pi0.py", line 289, in <module>
    train()
  File "/home/jeongeun/.pyenv/versions/3.10.2/envs/lerobot/lib/python3.10/site-packages/lerobot/configs/parser.py", line 226, in wrapper_inner
    response = fn(cfg, *args, **kwargs)
  File "/home/jeongeun/Dropbox/code/vla_prj/lerobot-mujoco-tutorial/train_pi0.py", line 110, in train
    cfg.validate()
  File "/home/jeongeun/.pyenv/versions/3.10.2/envs/lerobot/lib/python3.10/site-packages/lerobot/configs/train.py", line 101, in validate
    raise FileExistsError(
FileExistsError: Output directory ckpt/pi0_datawhale_eai already exists and resume is False. Please change your output directory so that ckpt/pi0_datawhale_eai is not overwritten.


## 第 3 步：部署


In [1]:
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
import numpy as np
from lerobot.common.datasets.utils import write_json, serialize_dict
from lerobot.common.policies.pi0.configuration_pi0 import PI0Config
from lerobot.common.policies.pi0.modeling_pi0 import PI0Policy
from lerobot.configs.types import FeatureType
from lerobot.common.datasets.factory import resolve_delta_timestamps
from lerobot.common.datasets.utils import dataset_to_policy_features
import torch
from PIL import Image
import torchvision

/home/jeongeun/.pyenv/versions/3.10.2/envs/lerobot/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 加载策略


In [2]:
device = 'cuda'

In [4]:
try:
    dataset_metadata = LeRobotDatasetMetadata("datawhale_eai_pnp_language", root='./demo_data_language')
except:
    dataset_metadata = LeRobotDatasetMetadata("datawhale_eai_pnp_language", root='./datawhale_eai_pnp_language')
features = dataset_to_policy_features(dataset_metadata.features)
output_features = {key: ft for key, ft in features.items() if ft.type is FeatureType.ACTION}
input_features = {key: ft for key, ft in features.items() if key not in output_features}
# Policies are initialized with a configuration class, in this case `DiffusionConfig`. For this example,
# we'll just use the defaults and so no arguments other than input/output features need to be passed.
# Temporal ensemble to make smoother trajectory predictions
cfg = PI0Config(input_features=input_features, output_features=output_features, chunk_size= 5, n_action_steps=5)
delta_timestamps = resolve_delta_timestamps(cfg, dataset_metadata)

In [5]:
# We can now instantiate our policy with this config and the dataset stats.
policy = PI0Policy.from_pretrained('./ckpt/pi0_datawhale_eai/checkpoints/last/pretrained_model', dataset_stats=dataset_metadata.stats)
# You can load the trained policy from hub if you don't have the resources to train it.
# policy = PI0Policy.from_pretrained("datawhale-eai/pi0_datawhale_eai", config=cfg, dataset_stats=dataset_metadata.stats)
policy.to(device)



Loading weights from local directory


PI0Policy(
  (normalize_inputs): Normalize(
    (buffer_observation_state): ParameterDict(
        (mean): Parameter containing: [torch.cuda.FloatTensor of size 6 (cuda:0)]
        (std): Parameter containing: [torch.cuda.FloatTensor of size 6 (cuda:0)]
    )
  )
  (normalize_targets): Normalize(
    (buffer_action): ParameterDict(
        (mean): Parameter containing: [torch.cuda.FloatTensor of size 7 (cuda:0)]
        (std): Parameter containing: [torch.cuda.FloatTensor of size 7 (cuda:0)]
    )
  )
  (unnormalize_outputs): Unnormalize(
    (buffer_action): ParameterDict(
        (mean): Parameter containing: [torch.cuda.FloatTensor of size 7 (cuda:0)]
        (std): Parameter containing: [torch.cuda.FloatTensor of size 7 (cuda:0)]
    )
  )
  (model): PI0FlowMatching(
    (paligemma_with_expert): PaliGemmaWithExpertModel(
      (paligemma): PaliGemmaForConditionalGeneration(
        (vision_tower): SiglipVisionModel(
          (vision_model): SiglipVisionTransformer(
            (em

### 部署策略


In [15]:
from mujoco_env.y_env2 import SimpleEnv2
xml_path = './asset/example_scene_y2.xml'
PnPEnv = SimpleEnv2(xml_path, action_type='joint_angle')


-----------------------------------------------------------------------------
name:[Tabletop] dt:[0.002] HZ:[500]
 n_qpos:[31] n_qvel:[28] n_qacc:[28] n_ctrl:[10]
 integrator:[IMPLICITFAST]

n_body:[23]
 [0/23] [world] mass:[0.00]kg
 [1/23] [front_object_table] mass:[1.00]kg
 [2/23] [camera] mass:[0.00]kg
 [3/23] [camera2] mass:[0.00]kg
 [4/23] [camera3] mass:[0.00]kg
 [5/23] [link1] mass:[2.06]kg
 [6/23] [link2] mass:[3.68]kg
 [7/23] [link3] mass:[2.39]kg
 [8/23] [link4] mass:[1.40]kg
 [9/23] [link5] mass:[1.40]kg
 [10/23] [link6] mass:[0.65]kg
 [11/23] [camera_center] mass:[0.00]kg
 [12/23] [tcp_link] mass:[0.32]kg
 [13/23] [rh_p12_rn_r1] mass:[0.07]kg
 [14/23] [rh_p12_rn_r2] mass:[0.02]kg
 [15/23] [rh_p12_rn_l1] mass:[0.07]kg
 [16/23] [rh_p12_rn_l2] mass:[0.02]kg
 [17/23] [body_obj_mug_5] mass:[0.00]kg
 [18/23] [object_mug_5] mass:[0.08]kg
 [19/23] [body_obj_plate_11] mass:[0.00]kg
 [20/23] [object_plate_11] mass:[0.10]kg
 [21/23] [body_obj_mug_6] mass:[0.00]kg
 [22/23] [object_mug

In [ ]:
from torchvision import transforms
# Approach 1: Using torchvision.transforms
def get_default_transform(image_size: int = 224):
    """
    Returns a torchvision transform that:
     Converts to a FloatTensor and scales pixel values [0,255] -> [0.0,1.0]
    """
    return transforms.Compose([
        transforms.ToTensor(),  # PIL [0–255] -> FloatTensor [0.0–1.0], shape C×H×W
    ])

In [17]:
step = 0
PnPEnv.reset(seed=0)
policy.reset()
policy.eval()
save_image = True
IMG_TRANSFORM = get_default_transform()
while PnPEnv.env.is_viewer_alive():
    PnPEnv.step_env()
    if PnPEnv.env.loop_every(HZ=20):
        # Check if the task is completed
        success = PnPEnv.check_success()
        if success:
            print('Success')
            # Reset the environment and action queue
            policy.reset()
            PnPEnv.reset()
            step = 0
            save_image = False
        # Get the current state of the environment
        state = PnPEnv.get_joint_state()[:6]
        # Get the current image from the environment
        image, wirst_image = PnPEnv.grab_image()
        image = Image.fromarray(image)
        image = image.resize((256, 256))
        image = IMG_TRANSFORM(image)
        wrist_image = Image.fromarray(wirst_image)
        wrist_image = wrist_image.resize((256, 256))
        wrist_image = IMG_TRANSFORM(wrist_image)
        data = {
            'observation.state': torch.tensor([state]).to(device),
            'observation.image': image.unsqueeze(0).to(device),
            'observation.wrist_image': wrist_image.unsqueeze(0).to(device),
            'task': [PnPEnv.instruction],
        }
        # Select an action
        action = policy.select_action(data)
        action = action[0,:7].cpu().detach().numpy()
        # Take a step in the environment
        _ = PnPEnv.step(action)
        PnPEnv.render()
        step += 1
        success = PnPEnv.check_success()
        if success:
            print('Success')
            break

DONE INITIALIZATION
Success
DONE INITIALIZATION
Success
DONE INITIALIZATION
Success
DONE INITIALIZATION
Success
DONE INITIALIZATION
Success
DONE INITIALIZATION


In [18]:
# policy.push_to_hub(
#     repo_id='datawhale-eai/pi0_datawhale_eai',
#     commit_message='Add trained policy for PnP task',
# )

model.safetensors: 100%|██████████| 7.54G/7.54G [14:34<00:00, 8.61MB/s]  


CommitInfo(commit_url='https://huggingface.co/datawhale-eai/pi0_datawhale_eai/commit/85152b3cbefeb37f26d3d8682e2db187621894c3', commit_message='Add trained policy for PnP task', commit_description='', oid='85152b3cbefeb37f26d3d8682e2db187621894c3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datawhale-eai/pi0_datawhale_eai', endpoint='https://huggingface.co', repo_type='model', repo_id='datawhale-eai/pi0_datawhale_eai'), pr_revision=None, pr_num=None)

$\pi_0$ 之所以能在具身智能圈引发巨大轰动，是因为它极其优雅地解决了一个历史难题：**如何让拥有“互联网级智慧”的视觉语言大模型（VLM）学会“干粗活（高频连续的底层控制）”而不变笨。**

我们先用一张 Mermaid 流程图来看看它的数据流向和核心组件，然后再深入拆解。

### 📊 $\pi_0$ 架构工作流

代码段

```mermaid
graph TD
    %% Inputs
    subgraph Inputs [输入层 Inputs]
        IMG[视觉输入 Wrist/Base Camera]
        TXT[文本指令 Language Prompt]
        STATE[本体状态 State e.g. 6D关节角]
        NOISE[随机噪声动作 Noisy Action + 时间步 t]
    end

    %% Vision-Language Backbone
    subgraph Brain [PaliGemma 大脑：语义理解]
        SIGLIP[SigLIP 视觉编码器]
        PROJ[多模态投影器 Multi-modal Projector]
        LLM_MAIN[Gemma 语言模型 2048-dim]
    end

    %% Action Expert
    subgraph Cerebellum [Gemma Expert 小脑：运动控制]
        STATE_PROJ[状态投影 State Proj 32->1024]
        ACT_PROJ[动作/时间投影 Action/Time Proj]
        EXPERT_LLM[Gemma 动作专家模型 1024-dim]
    end

    %% Flow Matching Output
    subgraph Outputs [输出层 Outputs]
        OUT_PROJ[Action Out Proj 1024->32]
        FLOW[Flow Matching 向量场预测]
        ACT_CLEAN[去噪后的连续动作 Continuous Actions]
    end

    %% Connections
    IMG --> SIGLIP --> PROJ --> LLM_MAIN
    TXT --> LLM_MAIN

    STATE --> STATE_PROJ --> EXPERT_LLM
    NOISE --> ACT_PROJ --> EXPERT_LLM
    
    %% The magic interaction
    LLM_MAIN -.->|深层特征融合与交叉注意力| EXPERT_LLM

    EXPERT_LLM --> OUT_PROJ --> FLOW
    FLOW -->|ODE求解与去噪迭代| ACT_CLEAN
```

------

### 🧠 代码拆解：$\pi_0$ 到底包含了什么？

根据你提供的代码输出，$\pi_0$ 模型主要由以下三大模块构成：

#### 1. 互联网级的大脑：`PaliGemmaForConditionalGeneration`

这是模型的基础（Base Model），负责“看”和“想”。

- **眼睛 (`SiglipVisionModel`)**：将输入的图像切成 Patch，通过 27 层的 Vision Transformer 提取极其强大的视觉特征。
- **翻译官 (`PaliGemmaMultiModalProjector`)**：将 1152 维的视觉特征映射到 2048 维，让语言模型能“看懂”图像。
- **思考核心 (`GemmaForCausalLM` - 18层)**：处理文本指令（如“把苹果放进碗里”）和图像特征，理解环境的上下文。

#### 2. 运动控制的小脑：`gemma_expert`

这正是 $\pi_0$ 最核心的创新点！

在传统的 Vision-Language-Action (VLA) 模型中（比如 Google 的 RT-2），研究人员强行让主 LLM 把动作当成文本单词（Token）输出。这导致模型容易“灾难性遗忘”（学会了控制机器人，却不会回答普通问题了）。

- $\pi_0$ 引入了一个**专门的专家网络（18层，1024维度的轻量级 Gemma）**。
- 主脑 PaliGemma 负责高层语义的理解，然后把特征传递给这个 `gemma_expert`。专家网络结合机器人当前的物理状态（`state_proj`）和历史动作，专注于计算如何驱动电机。这种“脑体分离又融合”的设计极其巧妙。

#### 3. 连续生成的魔法：`PI0FlowMatching`

注意看代码外层的 `PI0FlowMatching`。模型并不是直接输出下一个动作的坐标，而是采用了**流匹配（Flow Matching）**。

- **什么是 Flow Matching？** 它和 Diffusion（扩散模型）原理类似，是一种连续时间的生成模型。你给它一个随机噪声动作，它通过输入的时间步 $t$ (`action_time_mlp_in`)，一步步预测一个“向量场（Vector Field）”，将噪声推向真实、平滑的专家动作分布。
- **为什么不用离散 Token？** 机器人的控制需要极高的精度和频率（比如 50Hz 到 100Hz）。把动作变成离散的词汇会损失精度。Flow Matching 能够原生地生成高维的、连续的、极其平滑的动作轨迹。

------

### 🚀 为什么 $\pi_0$ 效果这么好，影响这么广泛？

1. **破解了“语义”与“控制”的冲突（Alignment Tax）**

   过去，用极其庞大的互联网图文数据训练出的 VLM，在微调适配小规模的机器人运动数据时，很容易顾此失彼。$\pi_0$ 的 `PaliGemmaWithExpertModel` 结构，相当于给爱因斯坦（VLM）配了一个专业的赛车手替身（Expert）。爱因斯坦负责指挥方向，赛车手负责踩油门打方向盘，各司其职。

2. **极强的泛化能力（Generalist Policy）**

   因为底座是强大的 PaliGemma，$\pi_0$ 具备开箱即用的 Zero-shot 空间推理能力。你换一个颜色不同的碗，或者换一个背景，它的“大脑”依然能认出来，并稳定地将信号传给“小脑”执行抓取。

3. **支持多机器人平台（跨具身本体）**

   通过灵活的 `state_proj` 和 `action_in/out_proj` 线性层（代码中是 32 维特征转换），它可以轻松适配 6 自由度机械臂、7 自由度机械臂、甚至是灵巧手和四足机器狗，只需要更换极其轻量的外围投影层即可。

4. **Flow Matching 带来的丝滑控制**

   传统的 MSE（均方误差）回归在遇到“同一种任务有多种做法”时，会输出平均值（导致动作四不像）。Flow Matching 这种生成式策略能够捕捉“多峰分布”，让机器人做出的动作像人类一样自然、连贯，且具有极强的抗干扰能力。